In [0]:
%run ../transform/operaciones_replica

In [0]:
TABLA_ORIGEN = "platform_dev.silver_byma.operaciones_replica"
tabla_destino = "platform_dev.gold_byma.dim_canal"

dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("dimension_canal")

In [0]:
try:
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(f"La tabla {tabla_destino} no existe. Iniciando creación...")
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True
    elif carga_full:
        logger.warning(f"Carga full forzada manualmente para {tabla_destino}.")
    else:
        logger.info(f"La tabla {tabla_destino} existe.")
        tiene_datos = spark.sql(f"SELECT 1 FROM {tabla_destino} LIMIT 1").take(1) != []
        if tiene_datos:
            logger.info(f"La tabla {tabla_destino} contiene datos. Carga incremental activada.")
        else:
            carga_full = True
            logger.warning(f"La tabla {tabla_destino} está vacía. Forzando carga full.")
except Exception:
    logger.exception(f"Error al validar la tabla {tabla_destino}")
    raise

In [0]:
logger.info(f"Iniciando lectura de la fuente: {TABLA_ORIGEN}")
df_origen = spark.table(TABLA_ORIGEN)

if not carga_full:
    fila_control = spark.sql(f"SELECT ultima_fecha_procesada FROM platform_dev.bronce_byma.control_cargas WHERE proceso = '{tabla_destino}'").first()
    if fila_control is not None and fila_control["ultima_fecha_procesada"] is not None:
        ultima_fecha = fila_control["ultima_fecha_procesada"]
    else:
        ultima_fecha = "1970-01-01"
    
    logger.info(f"Filtrando particiones nuevas desde: {ultima_fecha}")
    df_base = df_origen.filter(F.col("fecha_particion") > F.lit(ultima_fecha))
else:
    df_base = df_origen

df_transformado = (
    df_base
    .select(F.col("origen").alias("nombre_canal"))
    .filter("origen IS NOT NULL")
    .distinct()
    .withColumn(
        "tipo_canal",
        F.when(F.col("nombre_canal") == "API", "API programatica")
         .when(F.col("nombre_canal") == "IOLnet", "Tradicional")
         .otherwise("Digital self-service")
    )
)

df_transformado.createOrReplaceTempView("v_canales_preparados")

In [0]:
cantidad_registros = df_transformado.count()
try:
    if carga_full:
        logger.warning(f"Carga full: Sobreescribiendo {tabla_destino}")
        spark.sql(f"""
            INSERT OVERWRITE {tabla_destino} (nombre_canal, tipo_canal, _created_at)
            SELECT nombre_canal, tipo_canal, current_timestamp() FROM v_canales_preparados
        """)
    else:
        if cantidad_registros == 0:
            logger.info("No se detectaron canales nuevos.")
        else:
            logger.info(f"Carga incremental: Aplicando MERGE para {cantidad_registros} registros.")
            spark.sql(f"""
                MERGE INTO {tabla_destino} AS target
                USING v_canales_preparados AS source
                ON target.nombre_canal = source.nombre_canal
                WHEN NOT MATCHED THEN
                  INSERT (nombre_canal, tipo_canal, _created_at)
                  VALUES (source.nombre_canal, source.tipo_canal, current_timestamp())
            """)
except Exception:
    logger.exception(f"Error en escritura de {tabla_destino}")
    raise

In [0]:
ultima_fecha_origen = df_origen.agg(F.max("fecha_particion").alias("max_fecha")).first()["max_fecha"]
if ultima_fecha_origen is not None and cantidad_registros > 0:
    spark.sql(f"""
        MERGE INTO platform_dev.bronce_byma.control_cargas AS target
        USING (SELECT '{tabla_destino}' AS proceso, DATE('{ultima_fecha_origen}') AS u_fecha) AS source
        ON target.proceso = source.proceso
        WHEN MATCHED THEN UPDATE SET target.ultima_fecha_procesada = source.u_fecha, target.fecha_actualizacion = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (proceso, ultima_fecha_procesada, fecha_actualizacion) VALUES (source.proceso, source.u_fecha, current_timestamp())
    """)
    logger.info("Watermark de canales actualizado.")